## Modeling

0. Index
1. Train the models
2. Evaluate the models
3. Compare models

### 1. Train the model
goal: 
    determine the best_params, cv_score and val_score of each model

In [29]:
import pandas as pd
from pathlib import Path
import os
import sys

In [30]:
# This adds the parent directory (project root) to the Python path
root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_path not in sys.path:
    sys.path.append(root_path)

In [31]:
from src.models.train import train_model

In [32]:
X_train_fe = pd.read_csv('../data/processed/X_train.csv')
X_val_fe = pd.read_csv('../data/processed/X_val.csv')
X_test_fe = pd.read_csv('../data/processed/X_test.csv')

Y_train = pd.read_csv('../data/processed/Y_train.csv').squeeze()
Y_val = pd.read_csv('../data/processed/Y_val.csv').squeeze()
Y_test = pd.read_csv('../data/processed/Y_test.csv').squeeze()

In [33]:
X_train_fe.head()

,sex,cp,thal,isquemia_score,est_stroke_volume,slope
0,1.0,0.0,3.0,2.600000,1.171875,1.0
1,1.0,0.0,3.0,1.000000,0.565385,2.0
2,1.0,0.0,3.0,1.666667,0.832000,2.0
3,1.0,0.0,3.0,1.666667,0.773810,2.0
4,1.0,0.0,3.0,0.000000,0.570886,2.0


In [34]:
print("Shapes: ")
print(f" X_train:  {X_train_fe.shape}")
print(f" X_val:  {X_val_fe.shape}")
print(f" X_test:  {X_test_fe.shape}")

Shapes: 
 X_train:  (180, 6)
 X_val:  (61, 6)
 X_test:  (61, 6)


In [35]:
result_lr = train_model( "logistic_regression", X_train_fe, Y_train, X_val_fe ,Y_val)
result_rf = train_model( "random_forest", X_train_fe, Y_train, X_val_fe ,Y_val)
result_xgb = train_model( "xgboost", X_train_fe, Y_train, X_val_fe ,Y_val)

In [36]:
result_lr

{'model': LogisticRegression(C=10.0, max_iter=1000, random_state=42),
 'scaler': StandardScaler(),
 'best_params': {'C': 10.0},
 'cv_score': np.float64(0.9332759287925697),
 'val_score': 0.8170995670995671}

In [37]:
result_rf

{'model': RandomForestClassifier(max_depth=5, n_estimators=300, n_jobs=1, random_state=42),
 'scaler': None,
 'best_params': {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 300},
 'cv_score': np.float64(0.9375870743034055),
 'val_score': 0.7965367965367965}

In [38]:
result_xgb

{'model': XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=None, device=None, early_stopping_rounds=None,
               enable_categorical=True, eval_metric='logloss',
               feature_types=None, feature_weights=None, gamma=None,
               grow_policy=None, importance_type=None,
               interaction_constraints=None, learning_rate=0.05, max_bin=None,
               max_cat_threshold=None, max_cat_to_onehot=None,
               max_delta_step=None, max_depth=2, max_leaves=None,
               min_child_weight=None, missing=nan, monotone_constraints=None,
               multi_strategy=None, n_estimators=200, n_jobs=None,
               num_parallel_tree=None, ...),
 'scaler': None,
 'best_params': {'learning_rate': 0.05,
  'max_depth': 2,
  'n_estimators': 200,
  'subsample': 1.0},
 'cv_score': np.float64(0.9438138544891641),
 'val_score': 0.823051948051948}

### 2. Evaluate the model
goal: 
    calculate indicators such as Accuracy, ROC-AUC, Precision (1), Recall, F1 ,CV ROC-AUC and Gap of each model, and determine which will go to production


In [39]:
from src.models.evaluate import evaluate_model

In [40]:
metrics_lr = evaluate_model( 
    "logistic_regression", 
    result_lr["model"], 
    X_test_fe, 
    Y_test, 
    result_lr["cv_score"], 
    result_lr["scaler"]
    )


───────────────────────────────────────────────────────
logistic_regression

───────────────────────────────────────────────────────
 Accuracy : 0.8197
 ROC-AUC : 0.8885
 Precision : 0.8667
 Recall : 0.7879
 F1 : 0.8254
 CV ROC-AUC : 0.9333 (train)
 Gap : 0.0447 -> good generalization

  Confusion Matrix:
    TN (no disease, correct)  : 24
    FP (no disease, wrong)    : 4  ← predicted disease, was healthy
    FN (disease, missed)      : 7  ← predicted healthy, had disease
    TP (disease, correct)     : 26

              precision    recall  f1-score   support

  No Disease       0.77      0.86      0.81        28
     Disease       0.87      0.79      0.83        33

    accuracy                           0.82        61
   macro avg       0.82      0.82      0.82        61
weighted avg       0.82      0.82      0.82        61



In [41]:
metrics_rf = evaluate_model( 
    "random_forest", 
    result_rf["model"],
    X_test_fe, 
    Y_test, 
    result_rf["cv_score"], 
    result_rf["scaler"]
    )


───────────────────────────────────────────────────────
random_forest

───────────────────────────────────────────────────────
 Accuracy : 0.8197
 ROC-AUC : 0.8745
 Precision : 0.8438
 Recall : 0.8182
 F1 : 0.8308
 CV ROC-AUC : 0.9376 (train)
 Gap : 0.0631 -> possible overfitting

  Confusion Matrix:
    TN (no disease, correct)  : 23
    FP (no disease, wrong)    : 5  ← predicted disease, was healthy
    FN (disease, missed)      : 6  ← predicted healthy, had disease
    TP (disease, correct)     : 27

              precision    recall  f1-score   support

  No Disease       0.79      0.82      0.81        28
     Disease       0.84      0.82      0.83        33

    accuracy                           0.82        61
   macro avg       0.82      0.82      0.82        61
weighted avg       0.82      0.82      0.82        61



In [42]:
metrics_xgb = evaluate_model(
     "xgboost", 
     result_xgb["model"], 
     X_test_fe, 
     Y_test, 
     result_xgb["cv_score"], 
     result_xgb["scaler"])


───────────────────────────────────────────────────────
xgboost

───────────────────────────────────────────────────────
 Accuracy : 0.8197
 ROC-AUC : 0.8939
 Precision : 0.8667
 Recall : 0.7879
 F1 : 0.8254
 CV ROC-AUC : 0.9438 (train)
 Gap : 0.0499 -> good generalization

  Confusion Matrix:
    TN (no disease, correct)  : 24
    FP (no disease, wrong)    : 4  ← predicted disease, was healthy
    FN (disease, missed)      : 7  ← predicted healthy, had disease
    TP (disease, correct)     : 26

              precision    recall  f1-score   support

  No Disease       0.77      0.86      0.81        28
     Disease       0.87      0.79      0.83        33

    accuracy                           0.82        61
   macro avg       0.82      0.82      0.82        61
weighted avg       0.82      0.82      0.82        61



### 3. Compare the model
goal: 
    table to show the performance of the models

In [43]:
from src.models.compare import compare_models

In [44]:
all_metrics = [metrics_lr, metrics_rf, metrics_xgb]
all_results = [result_lr, result_rf, result_xgb]

comparison_df = compare_models(all_metrics, all_results, X_test_fe, Y_test)


  MODEL COMPARISON — TEST SET
                     Accuracy  ROC-AUC  Precision  Recall      F1  CV Score     Gap
model                                                                              
xgboost                0.8197   0.8939     0.8667  0.7879  0.8254    0.9438  0.0499
logistic_regression    0.8197   0.8885     0.8667  0.7879  0.8254    0.9333  0.0447
random_forest          0.8197   0.8745     0.8438  0.8182  0.8308    0.9376  0.0631

  Best model by ROC-AUC: xgboost
  ✔ ROC curves saved → /Users/miluskapajuelo/Documents/Heart-Disease-Prediction-using-Machine-Learning-End-to-End-ML-Pipeline-/figures/roc_curves_all_models.png


In [45]:
results_map = {
    metrics_lr["model_name"]: result_lr,
    metrics_rf["model_name"]: result_rf,
    metrics_xgb["model_name"]: result_xgb,
}

metrics_map = {
    metrics_lr["model_name"]: metrics_lr,
    metrics_rf["model_name"]: metrics_rf,
    metrics_xgb["model_name"]: metrics_xgb,
}

In [46]:
# Identify the best model
best_idx     = comparison_df["ROC-AUC"].idxmax()
best_result  = results_map[best_idx]
best_metrics = metrics_map[best_idx]
best_result

{'model': XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=None, device=None, early_stopping_rounds=None,
               enable_categorical=True, eval_metric='logloss',
               feature_types=None, feature_weights=None, gamma=None,
               grow_policy=None, importance_type=None,
               interaction_constraints=None, learning_rate=0.05, max_bin=None,
               max_cat_threshold=None, max_cat_to_onehot=None,
               max_delta_step=None, max_depth=2, max_leaves=None,
               min_child_weight=None, missing=nan, monotone_constraints=None,
               multi_strategy=None, n_estimators=200, n_jobs=None,
               num_parallel_tree=None, ...),
 'scaler': None,
 'best_params': {'learning_rate': 0.05,
  'max_depth': 2,
  'n_estimators': 200,
  'subsample': 1.0},
 'cv_score': np.float64(0.9438138544891641),
 'val_score': 0.823051948051948}

In [47]:
print(f"Best model: {best_idx}")

Best model: xgboost


### 4. Explain results


In [48]:
from src.models.explain import explain_model

In [49]:
shap_importance = explain_model(
best_result["model"], 
X_test_fe,
Y_test
)




  Generating SHAP bar chart (global importance)...
  ✔ Saved → /Users/miluskapajuelo/Documents/Heart-Disease-Prediction-using-Machine-Learning-End-to-End-ML-Pipeline-/figures/shap_bar.png
  Generating SHAP beeswarm plot...
  ✔ Saved → /Users/miluskapajuelo/Documents/Heart-Disease-Prediction-using-Machine-Learning-End-to-End-ML-Pipeline-/figures/shap_beeswarm.png

  SHAP Importance Table:
          feature  mean_abs_shap share
   isquemia_score       1.273659 28.9%
               cp       1.101640 25.0%
est_stroke_volume       0.789866 17.9%
             thal       0.684842 15.6%
              sex       0.483994 11.0%
            slope       0.067684  1.5%

  Top feature 'isquemia_score' accounts for 28.9% of total SHAP importance.

  Force plot — patient 0:
    True label     : 0
    Predicted      : 0
    P(disease)     : 4.51%
  ✔ Saved → /Users/miluskapajuelo/Documents/Heart-Disease-Prediction-using-Machine-Learning-End-to-End-ML-Pipeline-/figures/shap_force_patient_0.png


In [50]:
from importlib.metadata import version

In [51]:
version("xgboost")  

'3.3.0'

In [52]:
from src.utils.helpers import save_pickle, save_json
from src.utils.config import  CFG, PROJECT_ROOT
import datetime

In [53]:
metadata = {
    "model_name": best_idx,
    "model_version": "1.7.26",
    "trained_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "decision_threshold": 0.5,
    "features": CFG["features"]["selected"],
    "metrics": {
        "roc_auc": best_metrics["roc-auc"],
        "recall": best_metrics["recall"],
    },
    "libraries":{
        "scikit-learn": "sklearn.1.9.0",
        "xgboost": "xgboost.3.3.0",
    },
    "author": "Jhoselyn Pajuelo Villanueva"

}

In [54]:
save_json(metadata, PROJECT_ROOT / "models" / "model_metadata.json")

  ✔ Saved  → /Users/miluskapajuelo/Documents/Heart-Disease-Prediction-using-Machine-Learning-End-to-End-ML-Pipeline-/models/model_metadata.json


In [55]:
save_pickle(best_result["model"],  PROJECT_ROOT / "models" / "best_model.pkl")
#save_pickle(best_result["scaler"], PROJECT_ROOT / "models" / "scaler.pkl")  - if LR is best model

  ✔ Saved  → /Users/miluskapajuelo/Documents/Heart-Disease-Prediction-using-Machine-Learning-End-to-End-ML-Pipeline-/models/best_model.pkl
